<b><font size="6" color="#E8800A">Week 14b · Support vectors for prices</font></b><br>
<b><font size="4">A tolerance has units</font></b><br>

SVR ignores errors inside a tube. Changing the target units changes that tube, so we compare euro and log-target models inside one training-only search.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Imports</font>](#imports)
* [<font color='#E8800A'>A tube, support vectors and a prediction</font>](#tube)
* [<font color='#E8800A'>Kernels: similarity instead of a straight line</font>](#kernels)
* [<font color='#E8800A'>The kernel trick in lifted coordinates</font>](#kernel-trick)
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>Start with the defaults</font>](#default)
* [<font color='#E8800A'>SVR settings</font>](#settings)
* [<font color='#E8800A'>The target transformation changes the meaning of tolerance</font>](#units)
* [<font color='#E8800A'>Nested cross-validation</font>](#nested)
* [<font color='#E8800A'>The final search and its winner</font>](#search)
* [<font color='#E8800A'>Predict with the fitted winner</font>](#prediction)
* [<font color='#E8800A'>The course board</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)
* [<font color='#E8800A'>Same-prediction polynomial lift: two-panel geometry</font>](#polynomial-lift-geometry)

</div>

# <font color='#E8800A'>Imports</font> <a class="anchor" id="imports"></a>
[Back to TOC](#toc)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import joblib
from joblib import parallel_config
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate, train_test_split
from sklearn.svm import SVR, LinearSVR, NuSVR
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform, uniform, randint
import warnings
from sklearn.exceptions import ConvergenceWarning
from tqdm.auto import tqdm

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
from preprocessing import CachedSearchCV, FoldPreprocessor, PreparedEstimator, load_recipe, outer_folds, outer_splitter, prepared_folds
from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, show_log, record, result_context, results_before
from svm_scoring import expected_nonfinite_search_warning, ignore_unknown_category_warning, negative_mae_if_finite

random_state = 33
np.random.seed(random_state)
sns.set_theme(style='whitegrid')
set_matplotlib_formats('svg')
plt.rcParams.update({'figure.dpi': 150, 'figure.figsize': (8, 5)})
pd.set_option('display.max_colwidth', None)
regression_recipe_log = '../../logs/week_04_feature_work_regression_log.json'
ignore_unknown_category_warning()


# <font color='#E8800A'>A tube, support vectors and a prediction</font> <a class="anchor" id="tube"></a>
[Back to TOC](#toc)

For a linear SVR, $f(x)=w^\top x+b$. Here $x$ is one feature vector, $w$ contains its feature weights, $b$ is the intercept, and $f(x)$ is the predicted target. Training minimises

$$\frac12\lVert w\rVert^2+C\sum_i\max(0, |y_i-f(x_i)|-\varepsilon).$$

$y_i$ is training target $i$; $C$ weights violations against the penalty on the weights; $\varepsilon$ is the tube half-width. The loss is zero inside the tube and increases linearly outside. Rows with nonzero dual coefficients are support vectors. Points exactly on the tube may have zero coefficients, so geometry alone does not guarantee membership.

A kernel replaces the dot product. Prediction becomes $f(x)=b+\sum_{i\in SV}\beta_i K(x_i,x)$, where $SV$ indexes support vectors, $\beta_i$ is a signed fitted dual coefficient, and $K$ measures similarity. Prediction cost therefore depends on the support-vector count.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html'>sklearn.svm.SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale', degree=3, coef0=0.0)</a>

**Definition:**
Fit a function with an epsilon-insensitive tube.

**Main Parameters:**
- `C`: penalty for violations; `epsilon`: tube half-width in target units
- `kernel`: linear, rbf, poly or sigmoid; `gamma`: nonlinear similarity scale
- `degree`: polynomial degree; `coef0`: polynomial or sigmoid offset

**Main Attributes (after fitting):**
- `support_`: support row positions
- `support_vectors_`, `dual_coef_`, `intercept_`: terms of the fitted prediction

**Main Methods:**
- `fit(X, y)`: learn the function
- `predict(X)`: predict continuous targets
</div>

__Step 1:__ Prepare the eight toy training rows.

In [ ]:
toy_x = np.arange(8, dtype=float).reshape(-1, 1)
toy_y = np.array([0.0, 1.1, 1.7, 3.1, 4.0, 5.9, 6.1, 8.0])
print("Toy data:", np.column_stack((toy_x, toy_y)))


__Step 2:__ Fit the linear tube and inspect its supporting rows.

In [ ]:
toy = SVR(kernel="linear", C=10.0, epsilon=0.6).fit(toy_x, toy_y)
print("Support row positions:", toy.support_)


__Step 3:__ Predict one target and a curve on the toy axis.

In [ ]:
toy_query = np.array([[3.5]])
print("Support row positions:", toy.support_)
print("Prediction:", toy.predict(toy_query)[0])
toy_axis = np.linspace(0, 7, 150).reshape(-1, 1)
toy_prediction = toy.predict(toy_axis)
print("Curve shape:", toy_prediction.shape)


__Step 4:__ Inspect the fitted toy tube.

In [ ]:
fig, ax = plt.subplots()
ax.fill_between(toy_axis[:, 0], toy_prediction - toy.epsilon, toy_prediction + toy.epsilon,
                color=PLOT_BLUE, alpha=0.15, label="epsilon tube")
ax.plot(toy_axis[:, 0], toy_prediction, color=PLOT_BLUE, label="Prediction")
ax.scatter(toy_x[:, 0], toy_y, color="0.4", label="Training rows")
ax.scatter(toy_x[toy.support_, 0], toy_y[toy.support_], s=130, facecolors="none",
           edgecolors=PLOT_ORANGE, label="Support vectors")
ax.set(xlabel="Toy feature x", ylabel="Toy target", title="The tube and its supporting training rows")
ax.legend()
fig.tight_layout()
plt.show()

Only toy rows 2 and 7 are support vectors. The prediction at 3.5 is 3.83; the shaded tube extends 0.6 target units on either side.

### The loss, the slack and the two controls

The training loss is zero for any residual $y_i-f(x_i)$ inside the tube and grows linearly outside it: $\max(0,|y_i-f(x_i)|-\varepsilon)$. Writing the excess as slack, $\zeta_i$ when the target lies above the tube and $\zeta_i^*$ when it lies below, gives the primal form: minimise $\frac12w^\top w+C\sum_{i=1}^{n}(\zeta_i+\zeta_i^*)$ subject to $y_i-f(x_i)\le\varepsilon+\zeta_i$, $f(x_i)-y_i\le\varepsilon+\zeta_i^*$ and $\zeta_i,\zeta_i^*\ge0$. A row has at most one positive slack.

__Step 5:__ Plot the epsilon-insensitive loss against the residual.

In [ ]:
loss_epsilon = 0.6
loss_residual = np.linspace(-3, 3, 601)
loss_tube = np.maximum(0, np.abs(loss_residual) - loss_epsilon)
for loss_probe in [0.3, -0.6, 1.0, -2.0]:
    print(f"residual {loss_probe:+.1f}: loss {max(0.0, abs(loss_probe) - loss_epsilon):.1f}")
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.axvspan(-loss_epsilon, loss_epsilon, color=PLOT_BLUE, alpha=0.12, label="inside the tube")
ax.plot(loss_residual, np.abs(loss_residual), color="0.5", linestyle="--", label="absolute error")
ax.plot(loss_residual, loss_tube, color=PLOT_BLUE, linewidth=2.5, label="epsilon-insensitive loss")
ax.set(xlabel="residual y - f(x)", ylabel="loss", title="Epsilon-insensitive loss for epsilon = 0.6")
ax.legend(loc="upper center")
fig.tight_layout()
plt.show()

Residuals of 0.3 and -0.6 cost nothing, a residual of 1.0 costs 0.4 and a residual of -2.0 costs 1.4. Outside the tube the loss runs parallel to the absolute error, 0.6 below it.

__Step 6:__ Fit a linear SVR to thirty noisy rows and measure the slack of each row.

In [ ]:
tube_rng = np.random.default_rng(33)
tube_x = np.sort(tube_rng.uniform(0, 10, 30)).reshape(-1, 1)
tube_y = 1.2 * tube_x[:, 0] + 2 + tube_rng.normal(0, 1.2, 30)
print("Rows and feature shape:",len(tube_y),tube_x.shape)


__Step 7:__ Fit the tube on these rows.

In [ ]:
tube_model = SVR(kernel="linear", C=10.0, epsilon=0.8).fit(tube_x, tube_y)
print("Support vectors:",len(tube_model.support_))


__Step 8:__ Predict the training rows and inspect slack.

In [ ]:
tube_fitted = tube_model.predict(tube_x)
tube_slack = np.maximum(0, np.abs(tube_y - tube_fitted) - tube_model.epsilon)
print("rows:", len(tube_y), " support vectors:", len(tube_model.support_), " rows outside the tube:", int((tube_slack > 1e-3).sum()))
print("slack of the rows outside the tube:", tube_slack[tube_slack > 1e-3].round(3))
print(f"sum of slack: {tube_slack.sum():.3f}")

Of the 30 rows, 15 lie outside the tube and 17 are support vectors. The slacks of the rows outside the tube sum to 7.259; rows inside have zero slack and add nothing to the loss.

__Step 9:__ Draw the tube and the slack of every row outside it.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tube_axis = np.linspace(0, 10, 100).reshape(-1, 1)
tube_line = tube_model.predict(tube_axis)
ax.fill_between(tube_axis[:, 0], tube_line - tube_model.epsilon, tube_line + tube_model.epsilon, color=PLOT_BLUE, alpha=0.15, label="epsilon tube")
ax.plot(tube_axis[:, 0], tube_line, color=PLOT_BLUE, label="prediction")
tube_outside = tube_slack > 1e-3
ax.scatter(tube_x[~tube_outside, 0], tube_y[~tube_outside], color="0.4", label="inside the tube")
ax.scatter(tube_x[tube_outside, 0], tube_y[tube_outside], color=PLOT_ORANGE, label="outside the tube")
tube_edge = tube_fitted + np.sign(tube_y - tube_fitted) * tube_model.epsilon
ax.vlines(tube_x[tube_outside, 0], tube_edge[tube_outside], tube_y[tube_outside], color=PLOT_ORANGE, linewidth=2.5, label="slack")
ax.scatter(tube_x[tube_model.support_, 0], tube_y[tube_model.support_], s=150, facecolors="none", edgecolors="k", label="support vectors")
ax.set(xlabel="feature x", ylabel="target", title="Slack is the distance from a row outside the tube to the tube edge")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
fig.tight_layout()
plt.show()

Each orange segment is one slack from the table above, measured from the row to the tube edge. Every row with a segment is circled, because a row outside the tube always has a nonzero dual coefficient; the two other circled rows lie on the tube edge.

__Step 10:__ Fit the same data with three tube widths and three values of C.

In [ ]:
tube_settings = [(0.2, 10.0), (0.8, 10.0), (2.0, 10.0), (0.8, 0.02), (0.8, 0.1), (0.8, 1.0)]
print("Comparison settings:",tube_settings)


__Step 11:__ Fit the six linear tubes.

In [ ]:
tube_fits = [SVR(kernel="linear", C=c, epsilon=eps).fit(tube_x,tube_y) for eps,c in tube_settings]
print("Fitted tubes:",len(tube_fits))


__Step 12:__ Predict the same rows with all six tubes and inspect their errors.

In [ ]:
tube_rows = []
for (tube_eps, tube_c), tube_fit in zip(tube_settings, tube_fits):
    tube_resid = np.abs(tube_y - tube_fit.predict(tube_x))
    tube_rows.append({"epsilon": tube_eps, "C": tube_c, "support vectors": len(tube_fit.support_),
                      "rows outside the tube": int((tube_resid > tube_eps + 1e-3).sum()),
                      "slack sum": np.maximum(0, tube_resid - tube_eps).sum(),
                      "slope w": tube_fit.coef_[0, 0], "training MAE": tube_resid.mean()})
display(pd.DataFrame(tube_rows).round(3))

At C=10, widening the tube from epsilon 0.2 to 2.0 cuts the support vectors from 28 to 2 and the rows outside the tube from 26 to 0, while training MAE rises from 0.846 to 0.929. At epsilon 0.8, raising C from 0.02 to 1 lifts the slope from 0.811 to 1.196 and lowers the slack sum from 16.618 to 7.275; C=10 changes the fit little further (slack sum 7.259). The first row of panels varies epsilon and the second varies C.

__Step 13:__ Draw the six fits.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for ax, tube_fit, tube_row in zip(axes.ravel(), tube_fits, tube_rows):
    tube_line = tube_fit.predict(tube_axis)
    ax.fill_between(tube_axis[:, 0], tube_line - tube_row["epsilon"], tube_line + tube_row["epsilon"], color=PLOT_BLUE, alpha=0.15)
    ax.plot(tube_axis[:, 0], tube_line, color=PLOT_BLUE)
    ax.scatter(tube_x[:, 0], tube_y, color="0.4", s=25, label="training rows")
    ax.scatter(tube_x[tube_fit.support_, 0], tube_y[tube_fit.support_], s=120, facecolors="none", edgecolors=PLOT_ORANGE, label="support vectors")
    ax.set(title=f"epsilon={tube_row['epsilon']}, C={tube_row['C']}: {tube_row['support vectors']} support vectors")
for ax in axes[1]:
    ax.set_xlabel("feature x")
for ax in axes[:, 0]:
    ax.set_ylabel("target")
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

The tube follows epsilon in the top row, and the support vectors are the circled rows at or outside its edge. In the bottom row a small C leaves the line flatter than the rows, with a lower slope and more slack, while C=1 follows them.

# <font color='#E8800A'>Kernels: similarity instead of a straight line</font> <a class="anchor" id="kernels"></a>
[Back to TOC](#toc)

The tube above is a straight line in $x$. For a target that bends, a kernel replaces the dot product. The prediction is

$$f(x)=\sum_{i\in SV}(\alpha_i-\alpha_i^*)K(x_i,x)+b,$$

where $SV$ indexes the support vectors, $\alpha_i-\alpha_i^*$ is the signed dual coefficient stored in `dual_coef_`, $K$ measures the similarity between a support vector $x_i$ and the new row $x$, and $b$ is `intercept_`. Each support vector adds one weighted curve $(\alpha_i-\alpha_i^*)K(x_i,x)$.

Each kernel compares two rows $x$ and $x'$ with $p$ features through the dot product $\langle x,x'\rangle=\sum_{j=1}^{p}x_jx'_j$ and the squared distance $\lVert x-x'\rVert^2=\sum_{j=1}^{p}(x_j-x'_j)^2$.

| Kernel | Formula | Symbols | What it measures |
|---|---|---|---|
| `linear` | $\langle x,x'\rangle$ | none | the dot product alone |
| `poly` | $(\gamma\langle x,x'\rangle+r)^d$ | $\gamma>0$ is `gamma`, $r$ is `coef0`, $d$ is `degree` | the dot product, shifted by $r$ and raised to the power $d$ |
| `rbf` | $\exp(-\gamma\lVert x-x'\rVert^2)$ | $\gamma>0$ is `gamma` | the distance between the rows, and nothing else |
| `sigmoid` | $\tanh(\gamma\langle x,x'\rangle+r)$ | $\gamma$ is `gamma`, $r$ is `coef0` | the dot product, shifted by $r$ and squashed into $(-1,1)$ |

__Step 14:__ Build a one-feature nonlinear target.

In [ ]:
kernel_toy_rng = np.random.default_rng(33)
kernel_toy_x = np.sort(5 * kernel_toy_rng.random((40, 1)), axis=0)
kernel_toy_y = np.sin(kernel_toy_x).ravel()
kernel_toy_y[::5] += 3 * (0.5 - kernel_toy_rng.random(8))
kernel_toy_axis = np.linspace(0, 5, 401).reshape(-1, 1)
print("rows:", len(kernel_toy_y), " target range:", kernel_toy_y.min().round(3), "to", kernel_toy_y.max().round(3))

The sine target has 40 rows between 0 and 5. Eight of them are pushed away from the curve by noise, which stretches the target range to -1.468 and 1.878.

The four-kernel comparison below uses one set of settings, with $\varepsilon=0.2$ throughout. The linear SVR has C=1. The poly SVR has C=1, $\gamma=0.5$, $r=1$ and $d=3$. The rbf SVR has C=10 and $\gamma=0.5$; its C is raised so that the curve can follow the sine shape. The sigmoid SVR has C=1, $\gamma=0.5$ and $r=1$. The separate kernel-trick example uses a homogeneous degree-2 polynomial kernel to make its feature map visible.

### The kernel trick in lifted coordinates <a class="anchor" id="kernel-trick"></a>

For one feature, the homogeneous degree-2 polynomial kernel is $K(x,x')=(xx')^2$. Its feature map is $\phi(x)=x^2$, because $\phi(x)\phi(x')=x^2{x'}^2=(xx')^2$. A linear model in this lifted coordinate has the form $f(x)=w\phi(x)+b=wx^2+b$, which is a curve in the original coordinate.

`SVR(kernel="poly", degree=2, gamma=1, coef0=0)` fits this model using kernel similarities. It does not need us to construct the lifted training matrix. We square the coordinates only to display the geometry, and use `predict` for both panels.

__Step 15:__ Fit the quadratic-kernel SVR on the same toy rows.

In [ ]:
kernel_settings = {"linear": dict(C=1.0, epsilon=0.2),
                   "poly": dict(C=1.0, epsilon=0.2, degree=3, gamma=0.5, coef0=1.0),
                   "rbf": dict(C=10.0, epsilon=0.2, gamma=0.5),
                   "sigmoid": dict(C=1.0, epsilon=0.2, gamma=0.5, coef0=1.0)}
print("Kernel settings:",list(kernel_settings))


__Step 16:__ Fit the homogeneous quadratic kernel.

In [ ]:
kernel_trick_svr = SVR(kernel="poly", degree=2, gamma=1.0, coef0=0.0,
                       C=1.0, epsilon=0.2).fit(kernel_toy_x, kernel_toy_y)
print("Support vectors:",len(kernel_trick_svr.support_))


__Step 17:__ Predict the array used in both polynomial geometry panels.

In [ ]:
kernel_trick_prediction = kernel_trick_svr.predict(kernel_toy_axis)
print("Shared prediction shape:",kernel_trick_prediction.shape)


The same fitted function is linear in $x^2$ and curved in $x$. This restricted quadratic model illustrates the feature map; it does not give the most flexible fit to a sine target.

<a id='polynomial-lift-geometry'></a>
__Step 18:__ Display the fitted line in the lifted coordinate and its curve in the original coordinate.

In [ ]:
fig, (ax_lifted, ax_original) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
ax_lifted.scatter(kernel_toy_x[:, 0] ** 2, kernel_toy_y, color="0.4", s=30, label="same toy rows")
ax_lifted.plot(kernel_toy_axis[:, 0] ** 2, kernel_trick_prediction, color=PLOT_BLUE, label="SVR prediction")
ax_lifted.set(xlabel="lifted coordinate phi(x) = x squared", ylabel="target", title="Linear in the lifted coordinate")
ax_original.scatter(kernel_toy_x[:, 0], kernel_toy_y, color="0.4", s=30, label="same toy rows")
ax_original.plot(kernel_toy_axis[:, 0], kernel_trick_prediction, color=PLOT_BLUE, label="SVR prediction")
ax_original.set(xlabel="original feature x", title="Curved in the original coordinate")
ax_lifted.legend()
ax_original.legend()
fig.tight_layout()
plt.show()


Both panels show identical predicted targets. Changing the horizontal coordinate from $x^2$ back to $x$ turns the straight fitted line into a quadratic curve. The kernel supplied the lifted inner products without explicitly adding a squared feature to the training rows.

__Step 19:__ Draw the RBF similarity against distance for four values of gamma.

In [ ]:
kernel_distance = np.linspace(0, 3, 301)
kernel_gammas = [0.1, 0.5, 2.0, 10.0]
for kernel_g in kernel_gammas:
    print(f"gamma={kernel_g}: similarity falls to 0.5 at distance {np.sqrt(np.log(2) / kernel_g):.3f}")
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for kernel_g, kernel_shade in zip(kernel_gammas, plt.cm.Blues(np.linspace(0.4, 1.0, len(kernel_gammas)))):
    ax.plot(kernel_distance, rbf_kernel(kernel_distance.reshape(-1, 1), [[0.0]], gamma=kernel_g)[:, 0], color=kernel_shade, linewidth=2.2, label=f"gamma={kernel_g}")
ax.axhline(0.5, color="0.6", linestyle=":")
ax.set(xlabel="distance ||x - x'||", ylabel="RBF similarity K", title="RBF similarity against distance for four gamma values")
ax.legend()
fig.tight_layout()
plt.show()

The RBF similarity falls to 0.5 at distance 2.633 for $\gamma=0.1$ and at 0.263 for $\gamma=10$. A larger gamma shrinks the reach of one support vector, so a fitted curve built from such terms can bend more sharply.

__Step 20:__ Fit one SVR per kernel on the one-feature target.

In [ ]:
kernel_fits = {kernel_name: SVR(kernel=kernel_name, **kernel_settings[kernel_name]).fit(kernel_toy_x, kernel_toy_y)
               for kernel_name in ["linear", "poly", "rbf", "sigmoid"]}
print("Fitted kernels:",list(kernel_fits))


__Step 21:__ Predict training targets and inspect kernel errors.

In [ ]:
display(pd.DataFrame([{"kernel": kernel_name, "support vectors": len(kernel_fit.support_),
                       "training MAE": np.abs(kernel_toy_y - kernel_fit.predict(kernel_toy_x)).mean()}
                      for kernel_name, kernel_fit in kernel_fits.items()]).round(4))

Training MAE is 0.4264 for linear with 26 support vectors, 0.2252 for poly with 14, 0.2671 for rbf with 24 and 0.9360 for sigmoid with 34. These results describe the fixed toy settings, rather than the best achievable fit for each kernel.

__Step 22:__ Draw the four fitted curves with their tubes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
for ax, (kernel_name, kernel_fit) in zip(axes.ravel(), kernel_fits.items()):
    kernel_line = kernel_fit.predict(kernel_toy_axis)
    ax.fill_between(kernel_toy_axis[:, 0], kernel_line - kernel_fit.epsilon, kernel_line + kernel_fit.epsilon, color=PLOT_BLUE, alpha=0.15, label="epsilon tube")
    ax.plot(kernel_toy_axis[:, 0], kernel_line, color=PLOT_BLUE, label="prediction")
    ax.scatter(kernel_toy_x[:, 0], kernel_toy_y, color="0.4", s=25, label="training rows")
    ax.scatter(kernel_toy_x[kernel_fit.support_, 0], kernel_toy_y[kernel_fit.support_], s=120, facecolors="none", edgecolors=PLOT_ORANGE, label="support vectors")
    ax.set(title=f"{kernel_name}: {len(kernel_fit.support_)} support vectors, MAE {np.abs(kernel_toy_y - kernel_fit.predict(kernel_toy_x)).mean():.3f}")
for ax in axes[1]:
    ax.set_xlabel("feature x")
for ax in axes[:, 0]:
    ax.set_ylabel("target")
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="lower center", ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

The linear tube is a straight line through a curved target. The poly and rbf curves bend with the sine, and the sigmoid curve follows the target least well. These are training errors on 40 artificial rows, so they show the shapes rather than rank the kernels.

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

__Step 23:__ Load the data, set aside the test rows and prepare the training folds.

In [ ]:
cars, regression_recipe = load_recipe("../../data/interim/cars4you.csv", regression_recipe_log)
X = cars.loc[:, regression_recipe["categorical"] + regression_recipe["numeric"]].copy()
y = cars["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
cars_prep = FoldPreprocessor(regression_recipe).fit(X_train, y_train)
X_train_ready = cars_prep.transform(X_train)
X_test_ready = cars_prep.transform(X_test)
cars_folds = outer_splitter("regression", random_state=random_state)
with parallel_config(backend="loky", initializer=ignore_unknown_category_warning):
    cars_prepared = prepared_folds(FoldPreprocessor(regression_recipe), cars_folds, X_train, y_train)
cars_log = []
sweep_tables = {}


# <font color='#E8800A'>Start with the defaults</font> <a class="anchor" id="default"></a>
[Back to TOC](#toc)

SVR() means C=1, epsilon=0.1, kernel='rbf', gamma='scale', degree=3 and coef0=0. On a euro target, a tolerance of 0.1 is ten cents. These defaults may underfit badly in these units; the euro-target reference below is the baseline for every later model.

__Step 24:__ Measure the euro-target reference on the prepared training folds.

In [ ]:
default_svr = SVR()
reference_train, reference_validation = [], []
for fold in cars_prepared:
    reference_fold = clone(default_svr).fit(fold["X_fit"], fold["y_fit"])
    reference_train.append(mean_absolute_error(fold["y_fit"], reference_fold.predict(fold["X_fit"])))
    reference_validation.append(mean_absolute_error(fold["y_score"], reference_fold.predict(fold["X_score"])))
reference_train = np.array(reference_train)
reference_validation = np.array(reference_validation)
default_estimator = PreparedEstimator(FoldPreprocessor(regression_recipe), default_svr)
reference_scores = {"train_score": -reference_train, "test_score": -reference_validation}


__Step 25:__ Fit the reference on all training cars.

In [ ]:
fitted_reference = clone(default_estimator).fit(X_train, y_train)


The euro-target default evaluates $f(x)=b+\sum_{j\in SV}\beta_jK(s_j,x)$. Each $s_j$ is a support vector, $\beta_j$ is its signed dual coefficient and $b$ is the intercept. `predict` applies the fitted preprocessing and returns the price in euros.

__Step 26:__ Inspect the default representation and solver status.

In [ ]:
reference_svr = fitted_reference.estimator_
print("Support-vector matrix:", reference_svr.support_vectors_.shape)
print("Solver status:", reference_svr.fit_status_)


All 3200 training cars are support vectors, with 21 prepared features each. The ten-cent tube has not produced a sparse representation. Solver status 0 reports successful optimisation, which alone does not establish useful prediction accuracy.

__Step 27:__ Predict one held-out car price.

In [ ]:
reference_query = X_test.iloc[:1]
reference_prediction = fitted_reference.predict(reference_query)[0]
print("Prediction (EUR):", reference_prediction)
print("Actual price (EUR):", y_test.iloc[0])


The prediction is 14108.14 EUR for a car priced at 13500 EUR, an absolute error of 608.14 EUR.

__Step 28:__ Record the reference and its fold scores.

In [ ]:
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": len(cars_prepared), "name": "SVR()",
    "train": reference_train, "validation": reference_validation, "model": fitted_reference},
    X_test, y_test, "untuned", metric=mean_absolute_error, label="MAE")

# <font color='#E8800A'>SVR settings</font> <a class="anchor" id="settings"></a>
[Back to TOC](#toc)

Each comparison changes one argument of `SVR()`. The log records the lowest mean validation MAE. Test MAE describes each fitted row.

| Setting | Compared options |
|---|---|
| `epsilon` | 0.1, 100, 500, 1000, 3000, 10000 EUR |
| `C` | 1, 10, 100, 1000, 10000, 100000 |
| `kernel` | linear, rbf, poly, sigmoid |
| `gamma` | scale, 0.001, 0.01, 0.1, 1 |

### `SVR: epsilon`

The tube half-width has target units. Widening it permits larger errors without loss; inspect the support-vector count as well as MAE. C stays at its default value of 1.

The candidate list is `[0.1, 100, 500, 1000, 3000, 10000]`.

__Step 29:__ Change this control alone and measure its errors and support-vector counts on the prepared training folds.

In [ ]:
parameter = 'epsilon'
values = [0.1, 100.0, 500.0, 1000.0, 3000.0, 10000.0]
comparison_results = []
for value in tqdm(values, desc=parameter, leave=True, position=0):
    estimator = SVR(**{parameter: value})
    train_scores, validation_scores, support_counts = [], [], []
    for fold in cars_prepared:
        model = clone(estimator).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(mean_absolute_error(fold["y_fit"], model.predict(fold["X_fit"])))
        validation_scores.append(mean_absolute_error(fold["y_score"], model.predict(fold["X_score"])))
        support_counts.append(len(model.support_))
    comparison_results.append({"value": str(value), "configuration": repr(estimator),
        "train": train_scores, "validation": validation_scores,
        "train MAE": np.mean(train_scores), "cv MAE": np.mean(validation_scores),
        "support vectors": np.mean(support_counts), "estimator": estimator})
comparison_table = pd.DataFrame(comparison_results)

__Step 30:__ Display the comparison and its held-out errors.

In [ ]:
# Test errors are recorded after the training-fold experiment; selection uses cv MAE only.
for row in tqdm(comparison_results, desc='comparison results', leave=True, position=0):
    row["model"] = clone(row["estimator"]).fit(X_train_ready, y_train)
    row["test MAE"] = mean_absolute_error(y_test, row["model"].predict(X_test_ready))
comparison_table = pd.DataFrame(comparison_results)
sweep_tables[parameter] = comparison_table.copy()
display(comparison_table[["configuration", "train MAE", "cv MAE", "test MAE", "support vectors"]])


__Step 31:__ Record the validation-selected configuration.

In [ ]:
comparison_position = int(comparison_table["cv MAE"].idxmin())
comparison_winner = comparison_results[comparison_position]
cars_log[:] = [row for row in cars_log if row["validation MAE is"] != 'one setting changed: epsilon']
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": len(cars_prepared), "name": comparison_winner["configuration"],
    "train": comparison_winner["train"], "validation": comparison_winner["validation"],
    "model": comparison_winner["model"]}, X_test_ready, y_test,
    'one setting changed: epsilon', metric=mean_absolute_error, label="MAE")


__Step 32:__ Plot the recorded training, validation and held-out errors, then read the selected configuration diagnostics.

In [ ]:
fig, ax = plt.subplots()
positions = np.arange(len(comparison_table))
for column, color, marker, style in [("train MAE", "0.45", "o", "--"), ("cv MAE", PLOT_BLUE, "s", "-"), ("test MAE", PLOT_ORANGE, "^", "-")]:
    ax.plot(positions, comparison_table[column], marker=marker, linestyle=style, color=color, label=column)
ax.set(xticks=positions, xticklabels=comparison_table["value"], xlabel=parameter,
       ylabel="MAE (EUR)", title="One control changed; every other SVR setting is default")
ax.legend()
fig.tight_layout()
plt.show()
print("Validation-selected value:", comparison_winner["value"])
print("Training/validation gap (EUR):", comparison_winner["cv MAE"] - comparison_winner["train MAE"])

Epsilon changes sparsity without curing the default model: at C=1, validation MAE is 6862.81 EUR for epsilon=0.1 and 6862.41 EUR for epsilon=500, a difference of only 0.39 EUR. Widening the tube to 10000 reduces the mean support-vector count from 2560 to 582, but raises validation MAE to 7465.76 EUR. At the widest tube, the cheaper fitted representation gives worse predictions.

### `SVR: C`

C multiplies tube violations. Increasing it permits more complex fits when reducing those violations is worthwhile. All other controls keep their defaults; compare training and validation errors before claiming improvement.

The candidate list is `[1, 10, 100, 1000, 10000, 100000]`.

__Step 33:__ Change this control alone and measure its errors and support-vector counts on the prepared training folds.

In [ ]:
parameter = 'C'
values = [1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0]
comparison_results = []
for value in tqdm(values, desc=parameter, leave=True, position=0):
    estimator = SVR(**{parameter: value})
    train_scores, validation_scores, support_counts = [], [], []
    for fold in cars_prepared:
        model = clone(estimator).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(mean_absolute_error(fold["y_fit"], model.predict(fold["X_fit"])))
        validation_scores.append(mean_absolute_error(fold["y_score"], model.predict(fold["X_score"])))
        support_counts.append(len(model.support_))
    comparison_results.append({"value": str(value), "configuration": repr(estimator),
        "train": train_scores, "validation": validation_scores,
        "train MAE": np.mean(train_scores), "cv MAE": np.mean(validation_scores),
        "support vectors": np.mean(support_counts), "estimator": estimator})
comparison_table = pd.DataFrame(comparison_results)

__Step 34:__ Display the comparison and its held-out errors.

In [ ]:
# Test errors are recorded after the training-fold experiment; selection uses cv MAE only.
for row in tqdm(comparison_results, desc='comparison results', leave=True, position=0):
    row["model"] = clone(row["estimator"]).fit(X_train_ready, y_train)
    row["test MAE"] = mean_absolute_error(y_test, row["model"].predict(X_test_ready))
comparison_table = pd.DataFrame(comparison_results)
sweep_tables[parameter] = comparison_table.copy()
display(comparison_table[["configuration", "train MAE", "cv MAE", "test MAE", "support vectors"]])


__Step 35:__ Record the validation-selected configuration.

In [ ]:
comparison_position = int(comparison_table["cv MAE"].idxmin())
comparison_winner = comparison_results[comparison_position]
cars_log[:] = [row for row in cars_log if row["validation MAE is"] != 'one setting changed: C']
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": len(cars_prepared), "name": comparison_winner["configuration"],
    "train": comparison_winner["train"], "validation": comparison_winner["validation"],
    "model": comparison_winner["model"]}, X_test_ready, y_test,
    'one setting changed: C', metric=mean_absolute_error, label="MAE")


__Step 36:__ Plot the recorded training, validation and held-out errors, then read the selected configuration diagnostics.

In [ ]:
fig, ax = plt.subplots()
positions = np.arange(len(comparison_table))
for column, color, marker, style in [("train MAE", "0.45", "o", "--"), ("cv MAE", PLOT_BLUE, "s", "-"), ("test MAE", PLOT_ORANGE, "^", "-")]:
    ax.plot(positions, comparison_table[column], marker=marker, linestyle=style, color=color, label=column)
ax.set(xticks=positions, xticklabels=comparison_table["value"], xlabel=parameter,
       ylabel="MAE (EUR)", title="One control changed; every other SVR setting is default")
ax.legend()
fig.tight_layout()
plt.show()
print("Validation-selected value:", comparison_winner["value"])
print("Training/validation gap (EUR):", comparison_winner["cv MAE"] - comparison_winner["train MAE"])

C addresses the default model's underfitting: validation MAE falls from 6862.81 EUR at C=1 to 2431.28 EUR at C=100000. At that final value, training MAE is 2089.04 EUR and test MAE is 2458.70 EUR. The training-to-validation gap has grown to 342.25 EUR, but validation error is still improving at the boundary. This sweep has not located an interior optimum.

### `SVR: kernel`

Linear uses the dot product $\langle x,x'\rangle$, RBF uses local similarity $\exp(-\gamma\lVert x-x'\rVert^2)$, poly uses a power of the dot product, and sigmoid uses its hyperbolic tangent. Their default C and gamma need not suit them equally. A weak single-control result does not remove a kernel from the joint domain.

The candidate list is `["linear", "rbf", "poly", "sigmoid"]`.

__Step 37:__ Change this control alone and measure its errors and support-vector counts on the prepared training folds.

In [ ]:
parameter = 'kernel'
values = ['linear', 'rbf', 'poly', 'sigmoid']
comparison_results = []
for value in tqdm(values, desc=parameter, leave=True, position=0):
    estimator = SVR(**{parameter: value})
    train_scores, validation_scores, support_counts = [], [], []
    for fold in cars_prepared:
        model = clone(estimator).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(mean_absolute_error(fold["y_fit"], model.predict(fold["X_fit"])))
        validation_scores.append(mean_absolute_error(fold["y_score"], model.predict(fold["X_score"])))
        support_counts.append(len(model.support_))
    comparison_results.append({"value": str(value), "configuration": repr(estimator),
        "train": train_scores, "validation": validation_scores,
        "train MAE": np.mean(train_scores), "cv MAE": np.mean(validation_scores),
        "support vectors": np.mean(support_counts), "estimator": estimator})
comparison_table = pd.DataFrame(comparison_results)

__Step 38:__ Display the comparison and its held-out errors.

In [ ]:
# Test errors are recorded after the training-fold experiment; selection uses cv MAE only.
for row in tqdm(comparison_results, desc='comparison results', leave=True, position=0):
    row["model"] = clone(row["estimator"]).fit(X_train_ready, y_train)
    row["test MAE"] = mean_absolute_error(y_test, row["model"].predict(X_test_ready))
comparison_table = pd.DataFrame(comparison_results)
sweep_tables[parameter] = comparison_table.copy()
display(comparison_table[["configuration", "train MAE", "cv MAE", "test MAE", "support vectors"]])


__Step 39:__ Record the validation-selected configuration.

In [ ]:
comparison_position = int(comparison_table["cv MAE"].idxmin())
comparison_winner = comparison_results[comparison_position]
cars_log[:] = [row for row in cars_log if row["validation MAE is"] != 'one setting changed: kernel']
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": len(cars_prepared), "name": comparison_winner["configuration"],
    "train": comparison_winner["train"], "validation": comparison_winner["validation"],
    "model": comparison_winner["model"]}, X_test_ready, y_test,
    'one setting changed: kernel', metric=mean_absolute_error, label="MAE")


__Step 40:__ Plot the recorded training, validation and held-out errors, then read the selected configuration diagnostics.

In [ ]:
fig, ax = plt.subplots()
positions = np.arange(len(comparison_table))
for column, color, marker, style in [("train MAE", "0.45", "o", "--"), ("cv MAE", PLOT_BLUE, "s", "-"), ("test MAE", PLOT_ORANGE, "^", "-")]:
    ax.plot(positions, comparison_table[column], marker=marker, linestyle=style, color=color, label=column)
ax.set(xticks=positions, xticklabels=comparison_table["value"], xlabel=parameter,
       ylabel="MAE (EUR)", title="One control changed; every other SVR setting is default")
ax.legend()
fig.tight_layout()
plt.show()
print("Validation-selected value:", comparison_winner["value"])
print("Training/validation gap (EUR):", comparison_winner["cv MAE"] - comparison_winner["train MAE"])

Kernel choice alone cannot compensate for C=1: linear has the lowest validation MAE here, 6618.43 EUR, compared with 6862.81 EUR for RBF. Its training error, 6613.59 EUR, is almost as large as its validation error. The narrow gap is evidence of underfitting. The nonlinear kernels still need to be considered with their joint C and gamma settings.

### `SVR: gamma`

For RBF, $K(x,z)=\exp(-\gamma\lVert x-z\rVert^2)$, where $x$ and $z$ are two prepared cars and the squared norm is their squared Euclidean distance. Small gamma spreads influence broadly; large gamma concentrates it locally. At C=1, changing reach may leave the model underfit throughout. The joint search can expose interactions that this sweep cannot.

The candidate list is `["scale", 0.001, 0.01, 0.1, 1]`.

__Step 41:__ Change this control alone and measure its errors and support-vector counts on the prepared training folds.

In [ ]:
parameter = 'gamma'
values = ['scale', 0.001, 0.01, 0.1, 1.0]
comparison_results = []
for value in tqdm(values, desc=parameter, leave=True, position=0):
    estimator = SVR(**{parameter: value})
    train_scores, validation_scores, support_counts = [], [], []
    for fold in cars_prepared:
        model = clone(estimator).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(mean_absolute_error(fold["y_fit"], model.predict(fold["X_fit"])))
        validation_scores.append(mean_absolute_error(fold["y_score"], model.predict(fold["X_score"])))
        support_counts.append(len(model.support_))
    comparison_results.append({"value": str(value), "configuration": repr(estimator),
        "train": train_scores, "validation": validation_scores,
        "train MAE": np.mean(train_scores), "cv MAE": np.mean(validation_scores),
        "support vectors": np.mean(support_counts), "estimator": estimator})
comparison_table = pd.DataFrame(comparison_results)

__Step 42:__ Display the comparison and its held-out errors.

In [ ]:
# Test errors are recorded after the training-fold experiment; selection uses cv MAE only.
for row in tqdm(comparison_results, desc='comparison results', leave=True, position=0):
    row["model"] = clone(row["estimator"]).fit(X_train_ready, y_train)
    row["test MAE"] = mean_absolute_error(y_test, row["model"].predict(X_test_ready))
comparison_table = pd.DataFrame(comparison_results)
sweep_tables[parameter] = comparison_table.copy()
display(comparison_table[["configuration", "train MAE", "cv MAE", "test MAE", "support vectors"]])


__Step 43:__ Record the validation-selected configuration.

In [ ]:
comparison_position = int(comparison_table["cv MAE"].idxmin())
comparison_winner = comparison_results[comparison_position]
cars_log[:] = [row for row in cars_log if row["validation MAE is"] != 'one setting changed: gamma']
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": len(cars_prepared), "name": comparison_winner["configuration"],
    "train": comparison_winner["train"], "validation": comparison_winner["validation"],
    "model": comparison_winner["model"]}, X_test_ready, y_test,
    'one setting changed: gamma', metric=mean_absolute_error, label="MAE")


__Step 44:__ Plot the recorded training, validation and held-out errors, then read the selected configuration diagnostics.

In [ ]:
fig, ax = plt.subplots()
positions = np.arange(len(comparison_table))
for column, color, marker, style in [("train MAE", "0.45", "o", "--"), ("cv MAE", PLOT_BLUE, "s", "-"), ("test MAE", PLOT_ORANGE, "^", "-")]:
    ax.plot(positions, comparison_table[column], marker=marker, linestyle=style, color=color, label=column)
ax.set(xticks=positions, xticklabels=comparison_table["value"], xlabel=parameter,
       ylabel="MAE (EUR)", title="One control changed; every other SVR setting is default")
ax.legend()
fig.tight_layout()
plt.show()
print("Validation-selected value:", comparison_winner["value"])
print("Training/validation gap (EUR):", comparison_winner["cv MAE"] - comparison_winner["train MAE"])

Gamma alone also leaves the default model underfit: the default scale setting has validation MAE 6862.81 EUR, while gamma=0.001 reaches 6963.96 EUR. Across all shown values, the training error remains close to validation error and both remain high. This sweep does not exhibit the high-gamma overfitting pattern that can appear after C permits a more flexible fit.

# Four regression families <a id='families'></a>
SVR uses kernel support vectors. LinearSVR solves a linear tube with liblinear; SGDRegressor fits a linear epsilon-insensitive model through stochastic updates. NuSVR learns its tube width through nu. The linear references scale targets within each fit. Compare their measured errors and warnings with SVR before allowing all four families into one joint search.

__Step 45:__ Measure, diagnose and log all four families on the same price folds.

In [ ]:
family_models = {
    "SVR": TransformedTargetRegressor(regressor=SVR()),
    "LinearSVR": TransformedTargetRegressor(regressor=LinearSVR(dual="auto", max_iter=20000, random_state=33), transformer=StandardScaler()),
    "NuSVR": TransformedTargetRegressor(regressor=NuSVR(C=10000, nu=0.5)),
    "SGDRegressor": TransformedTargetRegressor(regressor=SGDRegressor(loss="epsilon_insensitive", max_iter=5000, random_state=33), transformer=StandardScaler())}
family_rows = []
for family_name, family_model in family_models.items():
    family_train, family_validation, family_warnings = [], [], []
    for fold in cars_prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted_family = clone(family_model).fit(fold["X_fit"], fold["y_fit"])
        family_warnings.append(sum(issubclass(w.category, ConvergenceWarning) for w in caught))
        family_train.append(-negative_mae_if_finite(fitted_family, fold["X_fit"], fold["y_fit"]))
        family_validation.append(-negative_mae_if_finite(fitted_family, fold["X_score"], fold["y_score"]))
    full_family = clone(family_model).fit(X_train_ready, y_train)
    record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
        "folds": len(cars_prepared), "name": family_name + " family reference",
        "train": np.array(family_train), "validation": np.array(family_validation), "model": full_family},
        X_test_ready, y_test, "family comparison", metric=mean_absolute_error, label="MAE")
    regressor = full_family.regressor_
    family_rows.append({"family": family_name, "train MAE EUR": np.mean(family_train),
        "validation MAE EUR": np.mean(family_validation), "fold convergence warnings": sum(family_warnings),
        "test MAE EUR (descriptive)": -negative_mae_if_finite(full_family, X_test_ready, y_test),
        "support vectors": len(regressor.support_) if hasattr(regressor, "support_") else None,
        "prediction for first held-out car EUR": full_family.predict(X_test_ready[:1])[0],
        "actual price EUR": y_test.iloc[0], "iterations": getattr(regressor, "n_iter_", None)})
family_comparison = pd.DataFrame(family_rows)
family_comparison["validation difference from SVR EUR"] = family_comparison["validation MAE EUR"] - family_comparison.iloc[0]["validation MAE EUR"]
display(family_comparison)


# Target units change tolerance <a id='units'></a>
[Back to TOC](#toc)

Raw-price epsilon has euro units. Log-price epsilon describes proportional error. LinearSVR and SGDRegressor use a fold-fitted StandardScaler on the raw target, making their raw epsilon a standard-deviation tolerance; inverse transformation restores euros before scoring. SVR and NuSVR use euro targets directly. NuSVR sets nu rather than epsilon: nu controls the fraction of errors and support vectors, while the fitted tube adapts to target units. Every target transformation is fitted inside each training fold.

__Step 46:__ Write the euro epsilon list that the raw-price candidates below use, covering ten-cent precision and large business tolerances. Fill it with `[0.1, 100.0, 500.0, 1000.0, 3000.0, 10000.0]`; the domain declaration repeats these values.

In [ ]:
exercise_tolerances = ...
print(exercise_tolerances)

RandomizedSearchCV draws 60 trials from conditional distributions with seed 33. Positive scales use loguniform: multiplying C or alpha by ten should be as plausible at either end of its range. Categorical kernel, loss, penalty and learning-rate choices stay discrete. Each outer search and the final search repeat the same seeded draw. Families and target branches are sampled uniformly through conditional dictionaries, rather than by the number of possible numeric combinations. No measured score changes the domain. The l1_ratio draw affects elasticnet only, and power_t affects invscaling only; these shared SGD options are inactive for the other sampled penalties or learning schedules.

__Step 47:__ Declare conditional distributions for all four regression families.

In [ ]:
svm_param_distributions = []
for log_target in (False, True):
    target = {"estimator__func": [np.log if log_target else None],
              "estimator__inverse_func": [np.exp if log_target else None],
              "estimator__transformer": [None]}
    for family in (SVR, NuSVR):
        for kernel in ("linear", "rbf", "poly", "sigmoid"):
            branch = dict(target)
            branch.update({"estimator__regressor": [family(cache_size=256, max_iter=10000)],
                "estimator__regressor__kernel": [kernel],
                "estimator__regressor__C": loguniform(0.01, 100) if log_target else loguniform(10, 100000),
                "estimator__regressor__tol": loguniform(1e-4, 1e-2)})
            if family is SVR:
                branch["estimator__regressor__epsilon"] = loguniform(0.001, 0.3) if log_target else loguniform(10, 5000)
            else:
                branch["estimator__regressor__nu"] = uniform(0.1, 0.7)
            if kernel != "linear":
                branch["estimator__regressor__gamma"] = loguniform(1e-4, 0.3)
            if kernel in ("poly", "sigmoid"):
                branch["estimator__regressor__coef0"] = uniform(0, 2)
            if kernel == "poly":
                branch["estimator__regressor__degree"] = randint(2, 5)
            svm_param_distributions.append(branch)
    linear_target = dict(target)
    if not log_target:
        linear_target["estimator__transformer"] = [StandardScaler()]
    svm_param_distributions.append(dict(linear_target, **{
        "estimator__regressor": [LinearSVR(dual='auto', max_iter=20000, random_state=33)],
        "estimator__regressor__C": loguniform(0.001, 100),
        "estimator__regressor__epsilon": loguniform(0.001, 0.3),
        "estimator__regressor__loss": ["epsilon_insensitive", "squared_epsilon_insensitive"],
        "estimator__regressor__tol": loguniform(1e-5, 1e-2),
        "estimator__regressor__fit_intercept": [True, False]}))
    svm_param_distributions.append(dict(linear_target, **{
        "estimator__regressor": [SGDRegressor(max_iter=5000, random_state=33)],
        "estimator__regressor__loss": ["epsilon_insensitive", "squared_epsilon_insensitive"],
        "estimator__regressor__alpha": loguniform(1e-6, 0.1),
        "estimator__regressor__epsilon": loguniform(0.001, 0.3),
        "estimator__regressor__penalty": ["l2", "l1", "elasticnet"],
        "estimator__regressor__l1_ratio": uniform(0.05, 0.9),
        "estimator__regressor__learning_rate": ["constant", "invscaling", "adaptive"],
        "estimator__regressor__eta0": loguniform(1e-4, 0.03),
        "estimator__regressor__power_t": uniform(0.1, 0.6),
        "estimator__regressor__tol": loguniform(1e-5, 1e-2)}))
print("Conditional branches:", len(svm_param_distributions))


# <font color='#E8800A'>Nested cross-validation</font> <a class="anchor" id="nested"></a>
[Back to TOC](#toc)

Each outer training slice runs the same 60-candidate, three-fold inner search. Its outer validation slice scores the selected model, estimating the entire search procedure. The final configuration comes from a fresh inner search on all training rows. Repeated folds share observations, so paired differences are descriptive and their spread is not an independent-sample standard error.

Some log-target candidates can overflow when exp returns their predictions to euros. The shared scoring rule returns negative infinity if any prediction is nonfinite, and otherwise returns ordinary negative MAE. This makes that candidate ineligible while it stays in the domain and the sample, so every search ranks the same sixty candidates. Inner and outer scoring use the same rule. Counts remain visible; a search with no finite candidate or a nonfinite outer or final result is a failed procedure, not a reported MAE.

__Step 48:__ Construct the seeded random-search template.

In [ ]:
inner_folds = KFold(n_splits=3, shuffle=True, random_state=random_state)
base_svr = PreparedEstimator(FoldPreprocessor(regression_recipe),
    TransformedTargetRegressor(regressor=SVR()))
search_template = CachedSearchCV(RandomizedSearchCV(base_svr,
    param_distributions=svm_param_distributions, n_iter=60, random_state=33,
    scoring=negative_mae_if_finite, cv=inner_folds, n_jobs=2, error_score="raise"))
print("60 trials, three inner folds; all scores return to EUR")


__Step 49:__ Run the complete search inside every outer training fold and compare paired validation errors.

In [ ]:
# Failed candidates have score -inf; their standard deviation is undefined.
# Suppress only invalid arithmetic while CachedSearchCV computes that statistic.
with parallel_config(backend="loky", initializer=ignore_unknown_category_warning), expected_nonfinite_search_warning(), np.errstate(invalid="ignore"):
    nested_fold_results = [cross_validate(search_template, X_train, y_train,
        scoring=negative_mae_if_finite, cv=[split], return_train_score=True,
        return_estimator=True, n_jobs=1, error_score="raise")
        for split in tqdm(list(cars_folds.split(X_train, y_train)), desc='Nested CV: outer folds', unit='fold', leave=True, position=0)]
nested_scores = {key: np.concatenate([result[key] for result in nested_fold_results])
                 for key in nested_fold_results[0]}
nested_nonfinite_counts = [int(np.sum(~np.isfinite(fitted.cv_results_["mean_test_score"])))
                           for fitted in nested_scores["estimator"]]
assert all(np.all(np.isfinite(fitted.cv_results_["mean_test_score"]) | np.isneginf(fitted.cv_results_["mean_test_score"])) for fitted in nested_scores["estimator"]), "Unexpected NaN or positive infinity in candidate scores"
print("Nonfinite candidate counts in outer searches:", nested_nonfinite_counts)
assert all(np.isfinite(fitted.best_score_) for fitted in nested_scores["estimator"]), "No finite candidate in an outer search: failed procedure"
assert np.all(np.isfinite(nested_scores["test_score"])), "Nonfinite outer validation error: failed procedure"
assert np.all(np.isfinite(nested_scores["train_score"])), "Nonfinite outer training error: failed procedure"
nested_train = -nested_scores["train_score"]
nested_validation = -nested_scores["test_score"]
paired = pd.DataFrame({"default MAE": reference_validation, "search MAE": nested_validation})
paired["improvement EUR"] = paired["default MAE"] - paired["search MAE"]
display(paired)
print("Mean paired improvement (EUR):", paired["improvement EUR"].mean())
print("Outer MAE mean and sample SD:", nested_validation.mean(), nested_validation.std(ddof=1))


Fresh nested search has mean validation MAE 2476.89 EUR and sample SD 191.25 EUR across ten outer folds, compared with 6862.81 EUR for SVR(). Read paired fold differences in the table. Overlapping training sets make the spread descriptive.

# <font color='#E8800A'>The final search and its winner</font> <a class="anchor" id="search"></a>
[Back to TOC](#toc)

__Step 50:__ Run the final search on all training rows.

In [ ]:
with parallel_config(backend="loky", initializer=ignore_unknown_category_warning), expected_nonfinite_search_warning(), np.errstate(invalid="ignore"):
    search_template.fit(X_train, y_train)
search = search_template
final_nonfinite_count = int(np.sum(~np.isfinite(search.cv_results_["mean_test_score"])))
assert np.all(np.isfinite(search.cv_results_["mean_test_score"]) | np.isneginf(search.cv_results_["mean_test_score"])), "Unexpected NaN or positive infinity in candidate scores"
print("Nonfinite candidates in final search:", final_nonfinite_count)
print("Nonfinite candidate settings:", [search.cv_results_["params"][i] for i in np.flatnonzero(~np.isfinite(search.cv_results_["mean_test_score"]))])
assert np.any(np.isfinite(search.cv_results_["mean_test_score"])), "No finite candidate: failed procedure"
assert np.isfinite(search.best_score_), "Nonfinite selected score: failed procedure"
final_predictions = search.predict(X_test)
assert np.all(np.isfinite(final_predictions)), "Nonfinite held-out prediction: failed procedure"
winner_name = repr(search.best_estimator_.estimator_)
winner_name = " ".join(winner_name.split())
print("Chosen settings:", search.best_params_)
print("Inner selection MAE (EUR):", -search.best_score_)
print("Test MAE (EUR):", mean_absolute_error(y_test, final_predictions))


Rank five finite candidates by training-only inner CV. Held-out MAE describes these fitted candidates and does not change their order, the shortlist or the winner.

__Step 51:__ Read the five candidates with the highest inner scores.

In [ ]:
ranking = pd.DataFrame({"settings": search.cv_results_["params"],
    "inner score": search.cv_results_["mean_test_score"]})
ranking["family"] = [type(p["estimator__regressor"]).__name__ for p in ranking["settings"]]
ranking["target"] = ["log price" if p.get("estimator__func") is np.log else ("scaled price" if p.get("estimator__transformer") is not None else "raw EUR") for p in ranking["settings"]]
finite_ranking = ranking[np.isfinite(ranking["inner score"])].copy()
top_five = finite_ranking.nlargest(5, "inner score").copy()
top_five_test = []
for candidate in top_five["settings"]:
    diagnostic_model = clone(base_svr).set_params(**clone(candidate, safe=False)).fit(X_train, y_train)
    diagnostic_score = negative_mae_if_finite(diagnostic_model, X_test, y_test)
    top_five_test.append(-diagnostic_score if np.isfinite(diagnostic_score) else np.inf)
top_five["test MAE EUR (descriptive)"] = top_five_test
top_five["held-out status"] = ["finite" if np.isfinite(value) else "nonfinite prediction" for value in top_five_test]
top_five["inner MAE EUR"] = -top_five["inner score"]
display(top_five)


The final training-only search selects log-price RBF SVR, with inner MAE 2493.11 EUR and descriptive test MAE 2615.29 EUR. The added family and target columns make each ranked candidate identifiable; the held-out column does not change this order.

__Step 52:__ Measure the chosen configuration on the prepared folds.

In [ ]:
winner_train, winner_validation = [], []
for fold in cars_prepared:
    winner_fold = clone(search.best_estimator_.estimator_).fit(fold["X_fit"], fold["y_fit"])
    winner_train.append(-negative_mae_if_finite(winner_fold, fold["X_fit"], fold["y_fit"]))
    winner_validation.append(-negative_mae_if_finite(winner_fold, fold["X_score"], fold["y_score"]))
winner_train = np.array(winner_train)
winner_validation = np.array(winner_validation)
assert np.all(np.isfinite(winner_train)) and np.all(np.isfinite(winner_validation))


__Step 53:__ Save the supplied score record with its model, fold errors, and validation interpretation.

In [ ]:
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": len(cars_prepared), "name": winner_name,
    "train": winner_train, "validation": winner_validation, "model": search}, X_test, y_test,
    "winner's own score", metric=mean_absolute_error, label="MAE")


__Step 54:__ Record the model and its fold scores.

In [ ]:
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_train),
    "folds": len(cars_prepared), "name": f"joint search over {search.search.n_iter} sampled combinations",
    "train": nested_train, "validation": nested_validation, "model": search}, X_test, y_test,
    "nested estimate", metric=mean_absolute_error, label="MAE")

The winner's own cross-validation reuses a configuration selected from these training rows, so it can be optimistic. The nested score evaluates the complete selection procedure without exposing outer validation rows to inner ranking. Keep nonfinite candidates visible and ineligible; inspect their settings before interpreting the search.

# Predict with the fitted winner <a id='prediction'></a>
[Back to TOC](#toc)

The selected family may use support vectors or a coefficient vector. Calling predict on the complete fitted estimator applies feature preprocessing and restores target units before returning euros.

__Step 55:__ Predict one held-out price and save the deployable winner.

In [ ]:
winner_prepared = search.best_estimator_
winner_svr = winner_prepared.estimator_.regressor_
query = X_test.iloc[:1]
print("Winner:", winner_name)
if hasattr(winner_svr, "support_"):
    print("Support vectors:", len(winner_svr.support_), "of", len(y_train))
else:
    print("Coefficient shape:", winner_svr.coef_.shape)
print("Iterations:", getattr(winner_svr, "n_iter_", None))
print("Fit status:", getattr(winner_svr, "fit_status_", "not exposed"))
print("Prediction (EUR):", winner_prepared.predict(query)[0])
print("Actual price (EUR):", y_test.iloc[0])
results_dir = "../../results"
Path(results_dir).mkdir(exist_ok=True)
joblib.dump(winner_prepared, Path(results_dir) / "week_14_regression_model.joblib")
reloaded_winner = joblib.load(Path(results_dir) / "week_14_regression_model.joblib")
np.testing.assert_allclose(reloaded_winner.predict(X_test), winner_prepared.predict(X_test))


Use the exposed support count or coefficient shape to describe the selected representation. A small tolerance does not guarantee a sparse fitted model.

# <font color='#E8800A'>The course board</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

__Step 56:__ Display the accumulated model attempts.

In [ ]:
show_log(cars_log, label="MAE")

__Step 57:__ Compare the nested procedure with saved course results.

In [ ]:
context = result_context("../../data/interim/cars4you.csv", regression_recipe_log,
    X_train.index, X_test.index, outer_folds=outer_folds(cars_folds, X_train, y_train))
floor = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
board = Scoreboard("Cars4You", "MAE (EUR)", floor=floor, context=context)
for earlier in results_before(14, "regression", results_dir, context=context):
    board.record(*earlier)
board.record(14, winner_name, float(np.mean(nested_validation)), mean_absolute_error(y_test, search.predict(X_test)))
print(board.table())
board.save(results_dir, 14, "regression", baseline_cv=float(np.mean(reference_validation)),
    baseline_test=mean_absolute_error(y_test, fitted_reference.predict(X_test)))

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. Epsilon sets a tolerance in target units; changing the target changes those units.
2. C controls how strongly violations matter, while gamma controls nonlinear similarity.
3. A small training-validation gap can accompany underfitting.
4. The nested estimate assesses the search; the final fitted winner supplies predictions.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

Smola and Schölkopf (2004), A tutorial on support vector regression. Bergstra and Bengio (2012), Random Search for Hyper-Parameter Optimization, Journal of Machine Learning Research 13, 281–305. scikit-learn SVR and TransformedTargetRegressor documentation.

For the feature-map example, see scikit-learn [polynomial kernels](https://scikit-learn.org/stable/modules/metrics.html#polynomial-kernel) and the [kernel trick](https://scikit-learn.org/stable/modules/svm.html#kernel-functions).